In [1]:
import pandas as pd
import numpy as np
from google.colab import files

# 1. Load data and clean attack_cat values
raw = pd.read_csv('/content/UNSW_NB15_training-set.csv')
raw['attack_cat'] = raw['attack_cat'].str.strip()

SEED = 42

# 2. Count and drop duplicate rows excluding the 'id' column
dup_columns = [col for col in raw.columns if col != 'id']
duplicates_mask = raw.duplicated(subset=dup_columns, keep='first')
duplicate_count = duplicates_mask.sum()
total_raw_rows = len(raw)
duplicate_percentage = (duplicate_count / total_raw_rows) * 100

print(f"Number of duplicate rows (excluding 'id'): {duplicate_count}")
print(f"Percentage of duplicates: {duplicate_percentage:.2f}%")

clean = raw[~duplicates_mask].copy()

# 3. Stratified sampling with a minimum floor of 200 rows per category
target_sample_size = 10000
total_clean_rows = len(clean)
overall_fraction = target_sample_size / total_clean_rows

sampled_groups = []
for cat, group in clean.groupby('attack_cat'):
    cat_size = len(group)
    # Calculate raw target for this category
    target_cat_size = int(round(cat_size * overall_fraction))
    # Apply floor rule: at least 200 or all available rows
    final_cat_size = max(target_cat_size, min(200, cat_size))

    sampled_group = group.sample(n=final_cat_size, random_state=SEED)
    sampled_groups.append(sampled_group)

sample = pd.concat(sampled_groups)
# Shuffle the overall sample
sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)

# 4. Build summary table
raw_counts = raw['attack_cat'].value_counts()
clean_counts = clean['attack_cat'].value_counts()
sample_counts = sample['attack_cat'].value_counts()

raw_shares = raw['attack_cat'].value_counts(normalize=True) * 100
sample_shares = sample['attack_cat'].value_counts(normalize=True) * 100

# Combine into a single summary table
summary_df = pd.DataFrame({
    'count in raw': raw_counts,
    'count in clean': clean_counts,
    'count in sample': sample_counts,
    '% share in raw': raw_shares,
    '% share in sample': sample_shares
}).fillna(0)

summary_df.index.name = 'attack_cat'
summary_df = summary_df.reset_index()

# 5. Build processing log table
log_df = pd.DataFrame({
    'step': [
        'raw_rows',
        'duplicates_removed',
        'rows_after_dedup',
        'sample_rows',
        'seed',
        'floor_per_category'
    ],
    'value': [
        total_raw_rows,
        duplicate_count,
        len(clean),
        len(sample),
        SEED,
        200
    ]
})

# 6. Save dataframes to CSV
sample_file = 'unsw_nb15_sample.csv'
counts_file = 'category_counts_full.csv'
log_file = 'prep_log.csv'

sample.to_csv(sample_file, index=False)
summary_df.to_csv(counts_file, index=False)
log_df.to_csv(log_file, index=False)

print("CSVs generated successfully. Initiating downloads...")

# 7. Download files
for file in [sample_file, counts_file, log_file]:
    files.download(file)

Number of duplicate rows (excluding 'id'): 67601
Percentage of duplicates: 38.55%
CSVs generated successfully. Initiating downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>